In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/credit_default_clean.csv")

X = df.drop(columns=["DEFAULT", "SEX"])
y = df["DEFAULT"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(21000, 22) (9000, 22)
0.221 0.221


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score

# Scale features so the coefficients are comparable
logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42),
)
logreg.fit(X_train, y_train)

# Predicted probability of default (PD) on the test set
pd_test = logreg.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, pd_test)
print("AUC:", round(auc, 3))
print("Gini:", round(2 * auc - 1, 3))

AUC: 0.714
Gini: 0.427


In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier

# Gradient boosting handles non-linear effects and needs no scaling
gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)

# Predicted probability of default (PD) on the test set
pd_test_gb = gb.predict_proba(X_test)[:, 1]

auc_gb = roc_auc_score(y_test, pd_test_gb)
print("AUC:", round(auc_gb, 3))
print("Gini:", round(2 * auc_gb - 1, 3))

AUC: 0.776
Gini: 0.552


In [4]:
from scipy.stats import ks_2samp

def ks_statistic(y_true, pd_scores):
    """Max separation between the PD distributions of defaulters and non-defaulters."""
    bad = pd_scores[y_true == 1]
    good = pd_scores[y_true == 0]
    return ks_2samp(bad, good).statistic

print("KS logistic regression:", round(ks_statistic(y_test, pd_test), 3))
print("KS gradient boosting:", round(ks_statistic(y_test, pd_test_gb), 3))

KS logistic regression: 0.372
KS gradient boosting: 0.419
